<a href="https://colab.research.google.com/github/adrianamimo/dark-matter-ngc3198-simulation/blob/main/baryonic_matter_stability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

n = 1000
G = 6.67e-11
M = 6.6821e40
r_min = 3.086e19
r_max = 1.1e21
t = 1.5768e14
t_sub = t / 20
times = 4000
M_dark = 0
a = 9.7384e20
p_0 = 1.049e-22
h_R = 8.2696e19

plt.rcParams["animation.embed_limit"] = 200

np.random.seed(60)

angles = np.random.uniform(0, 360, n)
angles = np.radians(angles)

b = r_max / 5
radius = np.random.exponential(scale=b, size=n)
radius = np.clip(radius, r_min, r_max)
x = radius * np.cos(angles)
y = radius * np.sin(angles)
positions = np.column_stack((x, y))

M_visible_enclosed = M * (1 - np.exp(-radius / h_R) * (1 + radius / h_R))
v = np.sqrt((G * M_visible_enclosed)/ radius)

M_dark_enclosed = 4 * np.pi * p_0 * (a**3) * (np.log(1 + radius / a) - radius / (a + radius))
M_enclosed = M_visible_enclosed + M_dark_enclosed
v_dark = np.sqrt((G * M_dark_enclosed)/ radius)
v_total = np.sqrt(v**2 + v_dark**2)

vx = -v_total * np.sin(angles)
vy = v_total * np.cos(angles)
velocities = np.column_stack((vx, vy))

def data(frame):
  global positions, velocities
  for _ in range(20):
    radius = np.sqrt(positions[:, 0]**2 + positions[:, 1]**2)

    M_visible_enclosed_actual = M * (1 - np.exp(-radius / h_R) * (1 + radius / h_R))
    if M_dark != 0:
      M_dark_enclosed2 = 4 * np.pi * p_0 * (a**3) * (np.log(1 + radius / a) - radius / (a + radius))
      M_enclosed2= M_visible_enclosed_actual + M_dark_enclosed2
    else:
      M_enclosed2 = M_visible_enclosed_actual

    g = (G * M_enclosed2) / (radius**2)
    ax = -g * (positions[:, 0] / radius)
    ay = -g * (positions[:, 1] / radius)
    accelerations = np.column_stack((ax, ay))

    velocities += accelerations * t_sub
    positions += velocities * t_sub

    draw.set_offsets(positions)
    return draw,

fig, ax = plt.subplots()
ax.set_xlim(-r_max * 2, r_max * 2)
ax.set_ylim(-r_max * 2, r_max * 2)
ax.set_facecolor("k")
fig.set_facecolor("k")
ax.set_title("Simulation of Galaxy without Dark Matter", color="white")

draw = ax.scatter(positions[:, 0], positions[:, 1], s=1, c="white")

animation = animation.FuncAnimation(fig, data, frames=times, interval=10, cache_frame_data=False)
animation_html = HTML(animation.to_jshtml())
plt.close(fig)
display(animation_html)